# CLIP4Outfit — Notebook A: adaptación de dominio de CLIP sobre pares ítem-texto de Polyvore

**Qué hace este notebook:** parte de los pesos públicos de `openai/clip-vit-base-patch32` y
continúa entrenando ambas torres (imagen y texto) sobre pares `(imagen de prenda, texto de
prenda)` de Polyvore, con pérdida contrastiva. El resultado es un checkpoint de CLIP
especializado en vocabulario y visual de moda, que usaremos como inicialización del
CLIP4Outfit en los notebooks siguientes.

**Esto NO es todavía Text-to-Outfit** -- es solo adaptación de dominio a nivel de ítem
individual, el primer escalón de los tres que describe el paper.

**Decisiones propias documentadas** (el paper no da detalle de esta fase, solo dice que la
hicieron): learning rate bajo (1e-6) típico de fine-tuning de CLIP, *cosine schedule* con
*warmup*, *early stopping* por pérdida de validación, y pérdida multi-positivo (por la
duplicidad de texto a nivel de ítem, aunque aquí es mucho más leve -- 90,6% de textos únicos --
que a nivel de outfit).

**Requisitos:** `transformers`, `torch`, `torchvision`, `Pillow`. Si no tienes `transformers`
instalado: `pip install transformers`.


In [ ]:
import os, json, random, math
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm

from transformers import CLIPModel, CLIPTokenizerFast, get_cosine_schedule_with_warmup

POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"  # AJUSTA A TU RUTA LOCAL
IMAGES_DIR    = POLYVORE_PATH + "images/"
OUTPUT_DIR    = POLYVORE_PATH + "data/processed/"
ITEM_METADATA_PATH = POLYVORE_PATH + "polyvore_item_metadata.json"

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
CLIP_IMG_SIZE = 224
CLIP_MAX_TOKENS = 77  # límite de contexto de CLIP

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Construir pares (imagen de ítem, texto de ítem)

Reutilizamos `glove_text_features.csv` (ya generado por tu pipeline original) para la lista de
`item_id` con imagen disponible, y `polyvore_item_metadata.json` para el texto, con la misma
estrategia de *fallback* que ya usa el resto del proyecto: `title` -> `description` ->
`url_name`.

In [ ]:
df_items = pd.read_csv(OUTPUT_DIR + "glove_text_features.csv")
df_items["item_id"] = df_items["item_id"].astype(str)
valid_item_ids = set(df_items["item_id"])
print(f"Ítems con imagen disponible (según glove_text_features.csv): {len(valid_item_ids)}")

with open(ITEM_METADATA_PATH) as f:
    item_metadata = json.load(f)


def get_item_text(entry):
    for field in ("title", "description"):
        val = entry.get(field, "")
        if isinstance(val, str) and val.strip():
            return val.strip()
    url_name = entry.get("url_name", "")
    if isinstance(url_name, str) and url_name.strip():
        return url_name.strip()
    return ""


item_pairs = []
skipped_no_text = 0
for item_id in valid_item_ids:
    entry = item_metadata.get(item_id)
    if entry is None:
        continue
    text = get_item_text(entry)
    if not text:
        skipped_no_text += 1
        continue
    item_pairs.append({
        "item_id": item_id,
        "image_path": os.path.join(IMAGES_DIR, f"{item_id}.jpg"),
        "text": text,
    })

print(f"Pares (imagen, texto) utilizables: {len(item_pairs)}")
print(f"  Descartados por falta de texto: {skipped_no_text}")

# split train/valid a nivel de ítem (independiente de los splits de outfit)
random.Random(SEED).shuffle(item_pairs)
n_valid = max(1000, int(0.02 * len(item_pairs)))
valid_pairs = item_pairs[:n_valid]
train_pairs = item_pairs[n_valid:]
print(f"train={len(train_pairs)}  valid={len(valid_pairs)}")

## 2. Cargar CLIP (pesos públicos de OpenAI) y su tokenizer

In [ ]:
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
tokenizer = CLIPTokenizerFast.from_pretrained(CLIP_MODEL_NAME)

n_params = sum(p.numel() for p in clip_model.parameters())
print(f"Parámetros totales de CLIP: {n_params:,}")
print(f"logit_scale inicial (aprendible): {clip_model.logit_scale.exp().item():.2f}")

## 2b. Encontrar el `BATCH_SIZE` máximo real para tu GPU

En vez de fijar `BATCH_SIZE` a ciegas, lo medimos empíricamente: se duplica el batch hasta que
revienta por falta de memoria (`OutOfMemoryError`), y luego se afina con búsqueda binaria entre
el último que cupo y el primero que falló. Se usan tensores aleatorios (no datos reales) porque
lo único que importa aquí es el tamaño, no el contenido -- así no hay que esperar a que cargue
el DataLoader para esta medición.

**Nota:** el resultado depende de qué más tengas abierto en la GPU en ese momento (otros
procesos, notebooks). Ejecuta esto con la GPU lo más "limpia" posible, y vuelve a ejecutarlo si
cambias de máquina o si otro proceso empieza a compartir la GPU.

In [ ]:
# def find_max_batch_size(model, tokenizer, device, start_batch=8, max_batch=512,
#                          img_size=CLIP_IMG_SIZE, max_tokens=CLIP_MAX_TOKENS):
#     """Duplica el batch hasta que revienta por OOM, luego afina con búsqueda binaria."""
#     model.train()

#     def try_batch(bs):
#         try:
#             torch.cuda.empty_cache()
#             torch.cuda.reset_peak_memory_stats()
#             pixel_values = torch.randn(bs, 3, img_size, img_size, device=device)
#             input_ids = torch.randint(0, tokenizer.vocab_size, (bs, max_tokens), device=device)
#             attention_mask = torch.ones(bs, max_tokens, dtype=torch.long, device=device)

#             with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
#                 outputs = model(pixel_values=pixel_values, input_ids=input_ids,
#                                  attention_mask=attention_mask)
#                 loss = outputs.logits_per_image.diag().mean()  # pérdida ficticia, solo para medir memoria
#             loss.backward()
#             model.zero_grad()
#             peak_gb = torch.cuda.max_memory_allocated() / 1e9
#             return True, peak_gb
#         except torch.cuda.OutOfMemoryError:
#             torch.cuda.empty_cache()
#             return False, None

#     bs = start_batch
#     last_ok = None
#     while bs <= max_batch:
#         ok, peak_gb = try_batch(bs)
#         print(f"batch={bs:4d} -> {'OK' if ok else 'OOM'}" + (f"  (pico: {peak_gb:.2f} GB)" if ok else ""))
#         if not ok:
#             break
#         last_ok = bs
#         bs *= 2

#     if last_ok is None:
#         print("Ni el batch mínimo cabe -- necesitas gradient checkpointing o reducir el modelo.")
#         return start_batch

#     low, high = last_ok, bs
#     while high - low > 4:
#         mid = (low + high) // 2
#         ok, peak_gb = try_batch(mid)
#         print(f"batch={mid:4d} -> {'OK' if ok else 'OOM'}" + (f"  (pico: {peak_gb:.2f} GB)" if ok else ""))
#         if ok:
#             low = mid
#         else:
#             high = mid

#     safe_batch = max(4, int(low * 0.85))  # margen de seguridad del 15%
#     print(f"\nBatch size máximo estable encontrado: {low}")
#     print(f"Recomendado con margen de seguridad (15%): {safe_batch}")
#     return safe_batch


# if device.type == "cuda":
#     MICRO_BATCH_SIZE = find_max_batch_size(clip_model, tokenizer, device)
# else:
#     MICRO_BATCH_SIZE = 32  # sin GPU, un valor conservador cualquiera para CPU
#     print("Sin GPU detectada -- usando MICRO_BATCH_SIZE por defecto para CPU.")

# TARGET_EFFECTIVE_BATCH = 128  # el batch "virtual" que queremos simular via gradient accumulation
# GRAD_ACCUM_STEPS = max(1, TARGET_EFFECTIVE_BATCH // MICRO_BATCH_SIZE)
# print(f"\nMICRO_BATCH_SIZE={MICRO_BATCH_SIZE}  GRAD_ACCUM_STEPS={GRAD_ACCUM_STEPS}  "
#       f"-> batch efectivo real={MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS}")

In [ ]:
MICRO_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 4

## 3. Preprocesado de imagen fiel al de CLIP original

(Resize 224 + normalización específica de CLIP -- distinta de la de ImageNet que usa tu RE --
implementada directamente con `torchvision` en vez de con el `CLIPImageProcessor` de HuggingFace
por velocidad, pero con los mismos valores exactos.)

In [ ]:
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

clip_image_transform = transforms.Compose([
    transforms.Resize(CLIP_IMG_SIZE, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(CLIP_IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])


def load_clip_image(path):
    img = Image.open(path).convert("RGB")
    return clip_image_transform(img)

## 4. Dataset + DataLoader

La tokenización del texto se hace en el `collate_fn`, en batch, en vez de una vez por muestra
en `__getitem__` -- es más rápido con el tokenizer *fast* de HuggingFace.

In [ ]:
class ItemPairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        image = load_clip_image(p["image_path"])
        return image, p["text"]


def make_collate_fn(tokenizer):
    def collate_fn(batch):
        images, texts = zip(*batch)
        pixel_values = torch.stack(images)
        tok = tokenizer(list(texts), padding="max_length", truncation=True,
                         max_length=CLIP_MAX_TOKENS, return_tensors="pt")
        return pixel_values, tok["input_ids"], tok["attention_mask"], list(texts)
    return collate_fn


# MICRO_BATCH_SIZE y GRAD_ACCUM_STEPS ya se calcularon en la Sección 2b, a partir de una
# medición real de tu GPU -- no hace falta fijarlos aquí a mano.
collate_fn = make_collate_fn(tokenizer)
train_loader = DataLoader(ItemPairDataset(train_pairs), batch_size=MICRO_BATCH_SIZE, shuffle=True,
                           collate_fn=collate_fn, num_workers=0, pin_memory=(device.type == "cuda"))
valid_loader = DataLoader(ItemPairDataset(valid_pairs), batch_size=MICRO_BATCH_SIZE, shuffle=False,
                           collate_fn=collate_fn, num_workers=0, pin_memory=(device.type == "cuda"))

## 5. Pérdida contrastiva multi-positivo

Mismo patrón que en la Opción 1 (Notebook 2 v2): los pares del batch con texto idéntico se
tratan como positivos mutuos, no como negativos forzados. Aquí el efecto es pequeño (solo
9,4% de duplicidad) pero mantenemos el mismo mecanismo por coherencia y porque no cuesta nada
añadirlo.

In [ ]:
def masked_nce(logits, positive_mask):
    log_prob = F.log_softmax(logits, dim=1)
    pos_count = positive_mask.sum(dim=1).clamp(min=1.0)
    loss_per_row = -(log_prob * positive_mask).sum(dim=1) / pos_count
    return loss_per_row.mean()


def clip_contrastive_loss(logits_per_image, texts):
    """logits_per_image ya viene escalado por clip_model.logit_scale (convención estándar
    de CLIP: se multiplica por la escala, no se divide -- ver discusión sobre la Ec. 2 del
    paper de CLIP4Outfit)."""
    device_ = logits_per_image.device
    text_to_gid = {}
    group_ids = []
    for t in texts:
        if t not in text_to_gid:
            text_to_gid[t] = len(text_to_gid)
        group_ids.append(text_to_gid[t])
    group_ids = torch.tensor(group_ids, device=device_)

    positive_mask = (group_ids.unsqueeze(1) == group_ids.unsqueeze(0)).float()
    loss_i2t = masked_nce(logits_per_image, positive_mask)
    loss_t2i = masked_nce(logits_per_image.T, positive_mask.T)
    return (loss_i2t + loss_t2i) / 2

## 6. Bucle de entrenamiento -- mixed precision + gradient accumulation + cosine schedule

LR bajo (1e-6) porque estamos haciendo *fine-tuning* completo de un modelo ya muy capaz --
un LR alto aquí destruiría rápido la representación general que ya trae CLIP de fábrica antes
de que le dé tiempo a especializarse en moda.

In [ ]:
LEARNING_RATE = 1e-6
WEIGHT_DECAY  = 0.01
EPOCHS        = 8
WARMUP_RATIO  = 0.05
PATIENCE      = 3

optimizer = torch.optim.AdamW(clip_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps,
                                             num_training_steps=total_steps)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

CKPT_PATH_CLIP = POLYVORE_PATH + "checkpoints_clip4outfit/clip_finetuned_polyvore.pt"
os.makedirs(os.path.dirname(CKPT_PATH_CLIP), exist_ok=True)


@torch.no_grad()
def evaluate_clip_loss(loader):
    clip_model.eval()
    losses = []
    for pixel_values, input_ids, attention_mask, texts in loader:
        pixel_values = pixel_values.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = clip_model(pixel_values=pixel_values, input_ids=input_ids,
                                  attention_mask=attention_mask)
            loss = clip_contrastive_loss(outputs.logits_per_image, texts)
        losses.append(loss.item())
    clip_model.train()
    return sum(losses) / len(losses)


best_val_loss = float("inf")
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    clip_model.train()
    train_losses = []
    optimizer.zero_grad()

    for step, (pixel_values, input_ids, attention_mask, texts) in enumerate(tqdm(train_loader, desc=f"Época {epoch}")):
        pixel_values = pixel_values.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = clip_model(pixel_values=pixel_values, input_ids=input_ids,
                                  attention_mask=attention_mask)
            loss = clip_contrastive_loss(outputs.logits_per_image, texts) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        train_losses.append(loss.item() * GRAD_ACCUM_STEPS)

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            # Clamp de logit_scale a ln(100), práctica estándar del CLIP original (Radford
            # et al. 2021) para evitar que la temperatura aprendible se dispare sin control.
            with torch.no_grad():
                clip_model.logit_scale.clamp_(0, math.log(100))

    val_loss = evaluate_clip_loss(valid_loader)
    train_loss_mean = sum(train_losses) / len(train_losses)
    print(f"Época {epoch:2d} | train_loss={train_loss_mean:.4f} | val_loss={val_loss:.4f} | "
          f"logit_scale={clip_model.logit_scale.exp().item():.2f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "clip_state_dict": clip_model.state_dict(),
            "val_loss": best_val_loss,
        }, CKPT_PATH_CLIP)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping en época {epoch} (sin mejora en {PATIENCE} épocas)")
            break

print(f"\nMejor val_loss: {best_val_loss:.4f}  -- guardado en {CKPT_PATH_CLIP}")

## 7. Comprobación rápida (*smoke test*)

Antes de dar por bueno el checkpoint, una comprobación cualitativa mínima: coge un ítem al azar
del set de validación y mira si su propio texto es, de entre varios candidatos, el que mejor
puntúa contra su imagen. No sustituye una evaluación rigurosa, pero detecta fallos evidentes
antes de construir el resto de la arquitectura encima.

In [ ]:
@torch.no_grad()
def smoke_test(n_candidates=8):
    clip_model.eval()
    sample = random.sample(valid_pairs, n_candidates)
    target = sample[0]

    image = load_clip_image(target["image_path"]).unsqueeze(0).to(device)
    texts = [p["text"] for p in sample]
    tok = tokenizer(texts, padding="max_length", truncation=True,
                     max_length=CLIP_MAX_TOKENS, return_tensors="pt").to(device)

    with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
        outputs = clip_model(pixel_values=image, input_ids=tok["input_ids"],
                              attention_mask=tok["attention_mask"])
    sims = outputs.logits_per_image.squeeze(0).float().cpu()
    ranking = sims.argsort(descending=True)

    print(f"Imagen de referencia: item_id={target['item_id']}  texto real=\"{target['text']}\"")
    print("Ranking de candidatos (debería quedar el texto real en 1er lugar):")
    for rank, idx in enumerate(ranking.tolist(), start=1):
        marker = " <-- texto real" if idx == 0 else ""
        print(f"  {rank}. score={sims[idx]:.2f}  \"{texts[idx][:60]}\"{marker}")

    clip_model.train()


smoke_test()